In [ ]:
import os
import shutil
import random
import tensorflow as tf #type: ignore
import keras
from keras.models import Sequential # type: ignore
from keras.layers import Conv2D,MaxPooling2D,Flatten,Dense # type: ignore
from tensorflow.keras.preprocessing.image import ImageDataGenerator # type: ignore
from keras.models import load_model
import matplotlib.pyplot as mp
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from keras.models import load_model
from keras.preprocessing import image

In [2]:
BaseDIR = r"G:\Coding Languages\Artificial Intelligence\Datasets\Rice Image Dataset\Rice Image"
TrainDIR = r"G:\Coding Languages\Artificial Intelligence\Datasets\Rice Image Dataset\Train"
TestDIR = r"G:\Coding Languages\Artificial Intelligence\Datasets\Rice Image Dataset\Test"

In [3]:
os.listdir(BaseDIR)

['Arborio', 'Basmati', 'Ipsala', 'Jasmine', 'Karacadag']

In [17]:
for i in os.listdir(BaseDIR):
    Path = os.path.join(BaseDIR,i)
    if os.path.isdir(Path):
        for index,Fname in enumerate(os.listdir(Path),start = 1):
            OldPath = os.path.join(Path,Fname)
            ext = os.path.splitext(Fname)[1]
            NewFname = f"{index}{ext}"
            NewPath = os.path.join(Path,NewFname)
            os.rename(OldPath,NewPath)
print("All Files Successfully Renamed")

FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'G:\\Coding Languages\\Artificial Intelligence\\Datasets\\Rice Image Dataset\\Rice Image\\Arborio\\10.jpg' -> 'G:\\Coding Languages\\Artificial Intelligence\\Datasets\\Rice Image Dataset\\Rice Image\\Arborio\\2.jpg'

In [18]:
for i in os.listdir(BaseDIR):
    Path = os.path.join(BaseDIR,i)
    if os.path.isdir(Path):
        image = os.listdir(Path)
        random.shuffle(image)

        TrainCount = int(len(image) * 0.7)

        TrainImages = image[:TrainCount]
        TestImages = image[TrainCount:]

        os.makedirs(os.path.join(TrainDIR,i),exist_ok = True)
        os.makedirs(os.path.join(TestDIR,i),exist_ok = True)

        for image in TrainImages:
            shutil.copy(os.path.join(Path,image),os.path.join(TrainDIR,i,image))

        for image in TestImages:
            shutil.copy(os.path.join(Path,image),os.path.join(TestDIR,i,image))

print("Dataset Split Complete")

Dataset Split Complete


In [4]:
ImgSize = (300,300)
BatchSize = 32
TrainDataGen = ImageDataGenerator(rescale=1/255,
                                rotation_range=20,
                                width_shift_range=0.2,
                                height_shift_range=0.2,
                                shear_range=0.2,
                                zoom_range=0.2,
                                horizontal_flip=True
                                )
TrainGen = TrainDataGen.flow_from_directory(TrainDIR,target_size = ImgSize, batch_size = BatchSize, class_mode = 'categorical')
TestDataGen = ImageDataGenerator(rescale = 1/255)
TestGen = TestDataGen.flow_from_directory(TestDIR,target_size = ImgSize,batch_size = BatchSize,class_mode = 'categorical')

Found 52500 images belonging to 5 classes.
Found 22500 images belonging to 5 classes.


In [5]:
NumClasses = len(os.listdir(TrainDIR))  

In [6]:
NumClasses

5

In [7]:
df = keras.utils.image_dataset_from_directory(TrainDIR,image_size = (180,180),batch_size = BatchSize,label_mode = 'categorical')

Found 52500 files belonging to 5 classes.


In [8]:
def GetModel():
    Model = Sequential()
    Model.add(Conv2D(filters = 128,kernel_size = (3,3),activation = 'relu',input_shape = (300,300,3)))
    Model.add(MaxPooling2D(pool_size = (5,5)))

    Model.add(Conv2D(filters=  128, kernel_size = (3,3), activation = 'relu'))
    Model.add(MaxPooling2D(pool_size = (5,5)))

    Model.add(Conv2D(filters = 512,kernel_size = (3,3), activation = 'relu'))
    Model.add(MaxPooling2D(pool_size = (3,3), strides = 2))

    Model.add(Flatten())
    Model.add(Dense(256,activation = 'relu'))

    Model.add(Dense(NumClasses,activation = 'softmax'))

    return Model

In [9]:
Model = GetModel()

g:\Applications\Python 3.10.11\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
Model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 298, 298, 128)  │         3,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 59, 59, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 57, 57, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 11, 11, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 9, 9, 512)      │       590,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,840,197 (10.83 MB)

 Trainable params: 2,840,197 (10.83 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
weights,biases = Model.layers[0].get_weights()

In [13]:
len(biases),len(weights)

(128, 3)

In [10]:
Model.compile(optimizer = 'adam',loss = 'categorical_crossentropy',metrics = ['accuracy'])

In [11]:
TrainDataGen = TrainDataGen.flow_from_directory(TrainDIR,target_size = ImgSize,batch_size = BatchSize, class_mode = 'categorical')
TestGen = TestDataGen.flow_from_directory(TestDIR,target_size = ImgSize,batch_size = BatchSize,class_mode = 'categorical')

Found 52500 images belonging to 5 classes.
Found 22500 images belonging to 5 classes.


In [12]:
History = Model.fit(TrainGen,epochs = 4, validation_data = TestGen)

g:\Applications\Python 3.10.11\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/4
1641/1641 ━━━━━━━━━━━━━━━━━━━━ 5475s 3s/step - accuracy: 0.7225 - loss: 0.6338 - val_accuracy: 0.4696 - val_loss: 3.6715
Epoch 2/4
1641/1641 ━━━━━━━━━━━━━━━━━━━━ 4498s 3s/step - accuracy: 0.9321 - loss: 0.1783 - val_accuracy: 0.5700 - val_loss: 2.1806
Epoch 3/4
1641/1641 ━━━━━━━━━━━━━━━━━━━━ 4441s 3s/step - accuracy: 0.9710 - loss: 0.0814 - val_accuracy: 0.5781 - val_loss: 2.7199
Epoch 4/4
1641/1641 ━━━━━━━━━━━━━━━━━━━━ 5926s 4s/step - accuracy: 0.9793 - loss: 0.0623 - val_accuracy: 0.8439 - val_loss: 0.5969


In [15]:
Model.save('Rice Images.h5')

In [11]:
LoadModel = load_model("Rice Images.h5")

In [12]:
img = image.load_img(r"G:\Coding Languages\Artificial Intelligence\Datasets\Rice Image Dataset\Rice Image\Arborio\14.jpg",target_size = (300,300))
img = image.img_to_array(img)
img = np.expand_dims(img,axis = 0)
img = img/255

In [13]:
prediction = LoadModel(img)

In [14]:
print(prediction)

tf.Tensor([[4.3733220e-04 1.1120662e-08 9.6923470e-01 3.0327994e-02 5.0983781e-16]], shape=(1, 5), dtype=float32)


In [15]:
TH = 0.5
PredictedClasses = int(prediction[0][0]>TH)

ClassIndices = TrainGen.class_indices
labels = {v:k for k,v in ClassIndices.items()}
print("The Image is of a:",labels[PredictedClasses])

The Image is of a: Arborio
